In [ ]:
import os
import shutil
from google.colab import drive
from google import genai
from google.genai import types
from PIL import Image
from pydantic import BaseModel

# ==========================================
# ENTER YOUR API KEY HERE
# ==========================================
GEMINI_API_KEY = "YOUR_GEMINI_API_KEY_HERE"

# ==========================================
# SETUP GOOGLE DRIVE & DIRECTORIES
# ==========================================
# 1. Connect to your Google Drive
drive.mount('/content/drive')

# 2. Setup Folder Paths
BASE_PATH = "/content/drive/MyDrive/Class_10_Maths_Bank"
INBOX_DIR = os.path.join(BASE_PATH, "00_Inbox")
PROCESSED_DIR = os.path.join(BASE_PATH, "01_Processed")
CHAPTERS_DIR = os.path.join(BASE_PATH, "Chapters")

CHAPTER_LIST = [
    "Real Numbers",
    "Polynomials",
    "Pair of Linear Equations in Two Variables",
    "Quadratic Equations",
    "Arithmetic Progressions",
    "Triangles",
    "Coordinate Geometry",
    "Introduction to Trigonometry",
    "Some Applications of Trigonometry",
    "Circles",
    "Areas Related to Circles",
    "Surface Areas and Volumes",
    "Statistics",
    "Probability"
]

# Automatically create all folders if they don't exist yet!
os.makedirs(INBOX_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)
for chapter in CHAPTER_LIST:
    os.makedirs(os.path.join(CHAPTERS_DIR, chapter), exist_ok=True)

print("All Google Drive folders are ready!")

# 3. Define AI Output Structure
class QuestionItem(BaseModel):
    question_number: str
    chapter_name: str
    box_2d: list[int]  # [ymin, xmin, ymax, xmax] scaled 0-1000

class PaperAnalysis(BaseModel):
    questions: list[QuestionItem]

# 4. Main Slicing & Sorting Function
def process_question_papers():
    if not GEMINI_API_KEY or GEMINI_API_KEY == "YOUR_GEMINI_API_KEY_HERE":
        print("\nERROR: Please replace 'YOUR_GEMINI_API_KEY_HERE' with your actual Gemini API key near the top of the cell!")
        return

    client = genai.Client(api_key=GEMINI_API_KEY)
    files = [f for f in os.listdir(INBOX_DIR) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

    if not files:
        print("\nYour '00_Inbox' folder is currently empty!")
        print("Go to your Google Drive -> Class_10_Maths_Bank -> 00_Inbox, upload a question paper photo, and click Play again.")
        return

    for filename in files:
        filepath = os.path.join(INBOX_DIR, filename)
        print(f"\nScanning & slicing: {filename}...")
        img = Image.open(filepath)
        width, height = img.size

        prompt = f"""
        Detect every individual math question on this exam paper.
        Include the question number, all sub-parts, and any associated figure/diagram inside its bounding box.
        Add a small padding around the text so nothing is cut off.
        Classify each question into EXACTLY ONE of these Class 10 Maths chapters: {CHAPTER_LIST}.
        Return the bounding box coordinates as [ymin, xmin, ymax, xmax] scaled from 0 to 1000.
        """

        response = client.models.generate_content(
            model='gemini-3.8-flash',
            contents=[img, prompt],
            config=types.GenerateContentConfig(
                response_mime_type="application/json",
                response_schema=PaperAnalysis,
                temperature=0.1
            )
        )

        result = PaperAnalysis.model_validate_json(response.text)

        for idx, q in enumerate(result.questions):
            ymin, xmin, ymax, xmax = q.box_2d

            # Add a slight 1% safety margin so text edges never get clipped
            left = max(0, int(((xmin - 10) / 1000) * width))
            top = max(0, int(((ymin - 10) / 1000) * height))
            right = min(width, int(((xmax + 10) / 1000) * width))
            bottom = min(height, int(((ymax + 10) / 1000) * height))

            cropped_img = img.crop((left, top, right, bottom))
            if cropped_img.mode in ("RGBA", "P"):
                cropped_img = cropped_img.convert("RGB")

            chapter_folder = os.path.join(CHAPTERS_DIR, q.chapter_name)
            os.makedirs(chapter_folder, exist_ok=True)

            base_name = os.path.splitext(filename)[0]
            clean_q_num = "".join(c for c in q.question_number if c.isalnum())
            out_name = f"{base_name}_Q{clean_q_num}_{idx+1}.jpg"

            cropped_img.save(os.path.join(chapter_folder, out_name), quality=95)
            print(f"  -> Saved Question {q.question_number} into '{q.chapter_name}'")

        shutil.move(filepath, os.path.join(PROCESSED_DIR, filename))
        print(f"Done with {filename}! Moved original paper to '01_Processed'.")

process_question_papers()